# 🩺 第三十二天 · 大模型 API 入门（DeepSeek 结构化抽取）

**今日目标（约 1 小时）**：把「网页聊天」升级成「**代码调用 API**」，学会让 DeepSeek **按 JSON 格式输出实体**——这是赛题十三和 CHIP 2026 共用的发动机。

**为什么必须学 API**：比赛要处理 480 首方剂 / 200 篇文献，你不可能一条条复制到网页里。API = 让 Python 代码自动批量调用大模型。

**成本**：deepseek-chat 极便宜（约 1 元/百万 token），你还有免费额度，练手花不了几毛钱。

## 第 0 步 · 什么是 API（30 秒）

- **网页聊天** = 你手动打字、手动复制，一次一句；
- **API** = 程序里的「函数调用」：`client.chat.completions.create(...)` 把问题发出去、把答案收回来，能循环、能批量、能存文件。

DeepSeek 的 API 是 **OpenAI 兼容**的，所以直接用 `openai` 这个库，只改一行 `base_url` 就能调 DeepSeek。

## ⚠️ 第 1 步 · 安全存 API Key（重中之重！）

**你的仓库是公开的，key 一旦写进 notebook 并 push，就全网泄露、被人盗刷。**

正确做法（今天只做一次）：
1. 在 `week5/` 文件夹里新建一个文本文件，命名 **`api_key.txt`**；
2. 里面**只粘贴一行**你的 DeepSeek API Key（`sk-` 开头，不要有空格、引号）；
3. 保存。这个文件我**已经加进 `.gitignore`**，永远不会上传 GitHub。

> 🔑 去哪拿 key：[platform.deepseek.com](https://platform.deepseek.com) → 登录 → 左侧「API Keys」→ 创建。

In [1]:
from openai import OpenAI

# 从 api_key.txt 读 key（已 gitignore，不会泄露）
with open("api_key.txt", encoding="utf-8") as f:
    api_key = f.read().strip()

client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")

resp = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "你是一名中医古籍实体识别助手。"},
        {"role": "user", "content": "用一句话解释：什么是中药的『炮制』？"},
    ],
)
print(resp.choices[0].message.content)

中药的炮制是指根据临床用药和制剂要求，对中药材进行净制、切制或加辅料处理（如炒、炙、蒸、煅等）以降低毒性、增强疗效或改变药性的传统加工方法。


## 第 2 步 · 为什么要 JSON 输出（赛题的关键）

实体抽取的结果是**结构化数据**，不能是聊天式的散文。比赛要的是这种：

```json
{"entities": [{"text": "头痛", "type": "症状"}]}
```

两种拿到 JSON 的方法：
1. **提示词约束**：在 prompt 里写死「只输出 JSON，格式是…」——靠模型自觉；
2. **JSON 模式**：加参数 `response_format={"type": "json_object"}`——DeepSeek 强制只吐合法 JSON（更稳，今天用它）。

> ⚠️ 重要工程细节：赛题十三要求实体的「起止下标」。**别让模型直接算下标**（它算不准古文的字符位置），正确做法是——模型只输出 `text`，你再用代码 `原文.find(text)` 定位下标。这个 D33 实战会用到。

In [2]:
import json

# 中医古籍实体零样本抽取（赛题十三第一步实战）
sentence = "太阳病，头痛发热，汗出恶风，桂枝汤主之。"

prompt = f"""你是一名中医古籍实体识别专家。请从下面的古文中抽取这 10 类实体：
疾病、证型、症状、中药、方剂、治法、经络、穴位、炮制、病机。

要求：
1. 只输出 JSON，不要输出任何其他文字；
2. 格式：{{"entities": [{{"text": "实体原文", "type": "类别"}}]}}；
3. 每个实体的 text 必须严格来自原文，一字不差。

古文：
{sentence}
"""

resp = client.chat.completions.create(
    model="deepseek-chat",
    messages=[{"role": "user", "content": prompt}],
    response_format={"type": "json_object"},   # JSON 模式
)
result = json.loads(resp.choices[0].message.content)
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "entities": [
    {
      "text": "太阳病",
      "type": "疾病"
    },
    {
      "text": "头痛",
      "type": "症状"
    },
    {
      "text": "发热",
      "type": "症状"
    },
    {
      "text": "汗出",
      "type": "症状"
    },
    {
      "text": "恶风",
      "type": "症状"
    },
    {
      "text": "桂枝汤",
      "type": "方剂"
    }
  ]
}


## 第 3 步 · 观察结果（写这里）

预期会抽出：太阳病（疾病）、头痛/发热/汗出/恶风（症状）、桂枝汤（方剂）。

**你的观察**：
1. 模型抽对了几类？有没有漏抽或错抽？模型都做对了
2. 如果抽错了，你觉得怎么改 prompt 能修？补充类别定义

## ✅ D32 完成标准

- [ ] api_key.txt 建好且能读到
- [ ] 基础调用跑通（打印出回答）
- [ ] JSON 模式实体抽取跑通，结果是合法 JSON
- [ ] 观察写完（对/漏/错 + 改进思路）
- [ ] 保存（Cmd + S）

> 完成后喊我验收。**D33 预告**：自建 20 条古籍校验集 → DeepSeek 批量抽取 → 人工打分，跑通赛题十三「零样本基线」。